In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

from ember2018_pca4_runner import load_balanced_sample

In [2]:
path = "data/ember2018/train_ember_2018_v2_features.parquet"
qdf, numeric_feature_cols, dataset_summary = load_balanced_sample(
    Path(path), samples_per_class=100, random_state=42
)

print("Original shape:", (dataset_summary["rows"], dataset_summary["columns"]))
print(pd.Series(dataset_summary["label_counts"]))

Dataset schema: 799,912 rows, 2,381 numeric features


Scanning labels...


Loading 200 sampled rows in bounded feature batches...


  scanned 204,800/799,912 rows


  scanned 409,600/799,912 rows


  scanned 614,400/799,912 rows


Original shape: (799912, 2382)
-1    199992
0     299991
1     299929
dtype: int64


In [3]:
# The bounded loader already removed unlabeled rows and reproduced Sean's
# independent 100-benign/100-malware samples.
qX = qdf[numeric_feature_cols]
qy = qdf["Label"].astype(int)

print("QSVM sample shape:", qX.shape)
print("Label counts:")
print(qy.value_counts())

QSVM sample shape: (200, 2381)
Label counts:
Label
0    100
1    100
Name: count, dtype: int64


In [4]:
qX_train, qX_test, qy_train, qy_test = train_test_split(
    qX,
    qy,
    test_size=0.3,
    random_state=42,
    stratify=qy
)

N_QFEATURES = 4

imputer = SimpleImputer(strategy="median", keep_empty_features=True)
qX_train = imputer.fit_transform(qX_train)
qX_test = imputer.transform(qX_test)

scaler = StandardScaler()
qX_train = scaler.fit_transform(qX_train)
qX_test = scaler.transform(qX_test)

pca = PCA(n_components=N_QFEATURES, random_state=42)
qX_train = pca.fit_transform(qX_train)
qX_test = pca.transform(qX_test)

range_scaler = MinMaxScaler(feature_range=(-1, 1))
qX_train = range_scaler.fit_transform(qX_train)
qX_test = range_scaler.transform(qX_test)

print("qX_train shape:", qX_train.shape)
print("qX_test shape:", qX_test.shape)
print("Any NaN train:", np.isnan(qX_train).any())
print("Any NaN test:", np.isnan(qX_test).any())

qX_train shape: (140, 4)
qX_test shape: (60, 4)
Any NaN train: False
Any NaN test: False


In [5]:
feature_map = ZZFeatureMap(
    feature_dimension=N_QFEATURES,
    reps=2,
    entanglement="linear"
)

kernel = FidelityQuantumKernel(
    feature_map=feature_map,
    enforce_psd=True
)

model = SVC(kernel="precomputed")

/tmp/ipykernel_33127/3747766485.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


In [6]:
start = time.process_time()

K_train = kernel.evaluate(qX_train, qX_train)
K_train = np.asarray(K_train, dtype=float)
K_train = np.nan_to_num(K_train, nan=0.0, posinf=1.0, neginf=0.0)

model.fit(K_train, qy_train);

In [7]:
K_test = kernel.evaluate(qX_test, qX_train)
K_test = np.asarray(K_test, dtype=float)
K_test = np.nan_to_num(K_test, nan=0.0, posinf=1.0, neginf=0.0)

y_pred = model.predict(K_test)
decision_scores = model.decision_function(K_test)

end = time.process_time()

In [8]:
print("Accuracy:", accuracy_score(qy_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(qy_test, y_pred))
print("ROC AUC:", roc_auc_score(qy_test, decision_scores))
print("CPU time:", end - start)

print("\nClassification Report:")
print(classification_report(qy_test, y_pred, target_names=["Benign", "Malware"]))

print("\nConfusion Matrix:")
print(confusion_matrix(qy_test, y_pred))

Accuracy: 0.7666666666666667
Balanced accuracy: 0.7666666666666666
ROC AUC: 0.7411111111111112
CPU time: 80.491458249

Classification Report:
              precision    recall  f1-score   support

      Benign       0.81      0.70      0.75        30
     Malware       0.74      0.83      0.78        30

    accuracy                           0.77        60
   macro avg       0.77      0.77      0.77        60
weighted avg       0.77      0.77      0.77        60


Confusion Matrix:
[[21  9]
 [ 5 25]]
